# Regression để giải thích: thời tiết "đáng giá" bao nhiêu lượt thuê?

Trưởng vận hành Capital Bikeshare chuẩn bị kế hoạch điều xe và nhân sự theo thời tiết. Câu hỏi của anh ấy **không
phải** "ngày mai bao nhiêu lượt" (đó là dự báo), mà là:

> "Giữ mọi thứ khác như nhau, trời ấm hơn 5°C, ẩm hơn 10 điểm, gió mạnh hơn 10 km/h, hay mưa nhỏ thì lượt thuê
> thay đổi bao nhiêu? Ngày lễ thì sao?"

Đó là câu hỏi **giải thích** (explanatory): cần hệ số đọc được theo đơn vị business, có khoảng tin cậy, có biến
kiểm soát hợp lý; R² cao hay thấp không phải mục tiêu chính.

Dữ liệu: `day.csv` của Bike Sharing (731 ngày 2011-2012). Nhãn mùa theo Readme của UCI (1 = spring, 2 = summer,
3 = fall, 4 = winter); lưu ý từ module 08: mã 1 thực tế phủ khoảng 21/12-20/3, tức giai đoạn lạnh nhất năm.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/11-stats-experimentation/exercises/intermediate/02-bike-explanatory-regression -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

SEASONS = {1: "spring", 2: "summer", 3: "fall", 4: "winter"}            # theo Readme UCI
WEATHER = {1: "clear", 2: "mist", 3: "light_rain_snow", 4: "heavy_rain"}

FORMULA = ("cnt ~ temp_c + hum_pct + wind_kmh + C(weather, Treatment('clear'))"
           " + C(season_name, Treatment('spring')) + C(year) + workingday + holiday")

## Phần A. Chuẩn bị và fit model

`prepare(day) -> DataFrame`: bản sao của `day` thêm các cột:

| Cột | Cách tính |
| --- | --- |
| `temp_c` | `temp * 41` (°C) |
| `hum_pct` | `hum * 100` (%) |
| `wind_kmh` | `windspeed * 67` (km/h) |
| `season_name` | `season` ánh xạ qua `SEASONS` |
| `weather` | `weathersit` ánh xạ qua `WEATHER` |
| `year` | `2011 + yr` |

`fit_explanatory(df, formula=FORMULA, cov_type="HC3")`: OLS bằng `smf.ols(formula, df)`, fit với sai số chuẩn
robust theo `cov_type` (HC3 mặc định: không giả định phương sai sai số bằng nhau). Trả kết quả của statsmodels.
Nếu `cov_type == "HAC"` thì truyền thêm `cov_kwds={"maxlags": 7}` (sai số tương quan theo thời gian trong một tuần).

In [ ]:
def prepare(day: pd.DataFrame) -> pd.DataFrame:
    """Bản sao thêm temp_c, hum_pct, wind_kmh, season_name, weather, year."""
    return day.assign(
        temp_c=day["temp"] * 41, hum_pct=day["hum"] * 100, wind_kmh=day["windspeed"] * 67,
        season_name=day["season"].map(SEASONS), weather=day["weathersit"].map(WEATHER), year=2011 + day["yr"],
    )


def fit_explanatory(df: pd.DataFrame, formula: str = FORMULA, cov_type: str = "HC3"):
    """OLS với sai số chuẩn robust (HC3, hoặc HAC maxlags 7)."""
    model = smf.ols(formula, df)
    if cov_type == "HAC":
        return model.fit(cov_type="HAC", cov_kwds={"maxlags": 7})
    return model.fit(cov_type=cov_type)

In [ ]:
# explore: đọc day.csv, fit model đầy đủ, xem bảng hệ số
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure  # noqa: E402

day = prepare(pd.read_csv(ensure("bike-sharing-uci") / "day.csv", parse_dates=["dteday"]))
res = fit_explanatory(day)
print(f"n = {int(res.nobs)}, R² = {res.rsquared:.3f}, lượt thuê TB/ngày = {day.cnt.mean():,.0f}")
print(pd.DataFrame({"coef": res.params, "ci_low": res.conf_int()[0], "ci_high": res.conf_int()[1],
                    "p": res.pvalues}).round(3).to_string())

## Phần B. Đọc hệ số theo đơn vị business

`effect_table(res, units) -> DataFrame`: `units` là dict `tên hệ số -> số đơn vị` (ví dụ `{"temp_c": 5}` nghĩa là
"ấm hơn 5°C"). Mỗi khoá một dòng theo thứ tự của dict, index là tên hệ số, cột:
`unit`, `effect` (hệ số × unit), `ci_low`, `ci_high` (khoảng tin cậy 95% của hệ số × unit), `p`.

`coef_path(df, var, steps) -> Series`: `steps` là dict `nhãn -> công thức`; với mỗi công thức, fit bằng
`fit_explanatory(df, formula)` và lấy hệ số của `var`. Index là các nhãn theo thứ tự của dict, tên Series là `var`.
Dùng để thấy hệ số của nhiệt độ thay đổi thế nào khi lần lượt thêm biến kiểm soát (omitted variable bias).

`turning_point(res, var, var_sq) -> float`: với model có cả `var` và `var_sq` (bình phương), giá trị của `var` tại
đỉnh (hoặc đáy) của đường cong: $$x^* = -\beta_1 / (2\beta_2)$$.

In [ ]:
def effect_table(res, units: dict) -> pd.DataFrame:
    """Hệ số × số đơn vị kèm CI 95% và p, mỗi khoá của units một dòng."""
    ci = res.conf_int()
    rows = {}
    for term, unit in units.items():
        rows[term] = {"unit": unit, "effect": res.params[term] * unit, "ci_low": ci.loc[term, 0] * unit,
                      "ci_high": ci.loc[term, 1] * unit, "p": res.pvalues[term]}
    return pd.DataFrame.from_dict(rows, orient="index")


def coef_path(df: pd.DataFrame, var: str, steps: dict) -> pd.Series:
    """Hệ số của var qua từng công thức trong steps."""
    return pd.Series({label: fit_explanatory(df, formula).params[var] for label, formula in steps.items()},
                     name=var)


def turning_point(res, var: str, var_sq: str) -> float:
    """-beta_var / (2 * beta_var_sq)."""
    return float(-res.params[var] / (2 * res.params[var_sq]))

In [ ]:
# explore: hiệu ứng theo đơn vị người vận hành hiểu
UNITS = {"temp_c": 5, "hum_pct": 10, "wind_kmh": 10, "C(weather, Treatment('clear'))[T.mist]": 1,
         "C(weather, Treatment('clear'))[T.light_rain_snow]": 1, "holiday": 1, "workingday": 1}
effects = effect_table(res, UNITS)
print(effects.round(1).to_string())

labels = ["+5°C", "+10 điểm ẩm", "+10 km/h gió", "trời mù", "mưa/tuyết nhẹ", "ngày lễ", "ngày làm việc"]
fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(effects))
ax.errorbar(effects["effect"], y, xerr=[effects["effect"] - effects["ci_low"], effects["ci_high"] - effects["effect"]],
            fmt="o", color="#4c78a8", capsize=3)
ax.axvline(0, color="#333", lw=0.8)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlabel("thay đổi lượt thuê/ngày, giữ các yếu tố khác cố định (CI 95%, HC3)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Mưa nhỏ lấy đi ~1 900 lượt/ngày; ấm thêm 5°C thêm ~620 lượt", loc="left")
plt.show()

In [ ]:
# explore: hệ số nhiệt độ khi thêm dần biến kiểm soát, sai số chuẩn HAC, và đường cong nhiệt độ
STEPS = {
    "chỉ nhiệt độ": "cnt ~ temp_c",
    "+ năm": "cnt ~ temp_c + C(year)",
    "+ mùa": "cnt ~ temp_c + C(year) + C(season_name, Treatment('spring'))",
    "+ thời tiết": ("cnt ~ temp_c + C(year) + C(season_name, Treatment('spring')) + hum_pct + wind_kmh"
                    " + C(weather, Treatment('clear'))"),
    "đầy đủ": FORMULA,
}
print(coef_path(day, "temp_c", STEPS).round(1).to_string())

from statsmodels.stats.stattools import durbin_watson
hac = fit_explanatory(day, cov_type="HAC")
print(f"Durbin-Watson = {durbin_watson(res.resid):.2f} (2 là không tự tương quan)")
print(pd.DataFrame({"se_HC3": res.bse, "se_HAC": hac.bse}).loc[["temp_c", "hum_pct", "wind_kmh", "holiday"]].round(2))

quad = fit_explanatory(day, FORMULA.replace("temp_c +", "temp_c + I(temp_c ** 2) +", 1))
peak = turning_point(quad, "temp_c", "I(temp_c ** 2)")
print(f"đỉnh đường cong nhiệt độ: {peak:.1f}°C; {(day.temp_c > peak).mean():.0%} số ngày nóng hơn mức này; "
      f"R² {res.rsquared:.3f} -> {quad.rsquared:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(day.dteday, res.resid, s=5, alpha=0.5, color="#4c78a8")
ax.axhline(0, color="#333", lw=0.8)
ax.set_ylabel("residual (lượt)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Residual theo ngày: các ngày liền nhau lệch cùng chiều (tự tương quan)", loc="left")
plt.show()

## Đọc kết quả, rồi trả lời trưởng vận hành

1. Hệ số `temp_c` từ model chỉ có nhiệt độ tới model đầy đủ thay đổi thế nào? Biến nào "lấy đi" nhiều nhất? Vì
   sao model một biến **phóng đại** ảnh hưởng của nhiệt độ?
2. Đọc hệ số `holiday` và `workingday` theo câu "giữ các yếu tố khác cố định...". `workingday` có ý nghĩa thống kê
   không? (bài Basic 03 đã cho thấy vì sao: hai loại khách đi ngược chiều.)
3. Hệ số mùa: `winter` so với `spring` dương rất lớn. Có phải mùa đông ở Washington DC đông khách hơn mùa xuân?
   Mã mùa phủ những tháng nào?
4. Durbin-Watson bao nhiêu? HAC làm sai số chuẩn của nhiệt độ thay đổi ra sao? Kết luận nào còn đứng vững?
5. Đỉnh đường cong nhiệt độ ở khoảng bao nhiêu độ? Câu "ấm thêm 5°C thêm ~620 lượt" đúng cho mọi ngày không?
6. Vì sao hệ số của **thời tiết** đáng tin hơn để diễn giải gần như nhân quả, còn một hệ số như "số trạm mở mới"
   (nếu có) thì không?

Viết ba câu cho trưởng vận hành, mỗi câu một con số có khoảng tin cậy.

Xong thì mở app, làm phần **trắc nghiệm** của bài.